# Greenhouse design walkthrough

This notebook sizes a hydroponic lettuce greenhouse with the `hydroponics` package:
bench layout, supply-network hydraulics, pump selection and bill of materials.

**Vocabulary**

| term | meaning |
|---|---|
| NFT | *Nutrient Film Technique*: plants sit in holes along shallow channels; a thin film of nutrient solution flows through the channels by gravity and returns to a reservoir. |
| channel / profile | the plastic gutter that holds the plants (sold in 6 m bars; `PS55`, `PS85`... are profile designations with different hole spacing). |
| bench | a set of parallel channels on trestles. |
| bay | the span between two lines of greenhouse posts. |
| sector | an independent irrigation circuit with its own pump and reservoir. |
| manifold / lateral | the pipe with many outlets that runs along the greenhouse / the pipe that crosses a bay and feeds its benches. |
| TDH | total dynamic head: the pressure (in metres of water column, m) the pump has to deliver. |

All units are SI: m, m³/h, L/min, m of head, kW (and *cv*, the metric horsepower used in pump catalogues).

In [1]:
import warnings

import pandas as pd

from hydroponics import (
    Bench,
    BenchGroup,
    DesignInputs,
    DesignWarning,
    Greenhouse,
    HydraulicSettings,
    PumpStationSettings,
    christiansen_factor,
    design_greenhouse,
    hazen_williams_head_loss,
    plan_layout,
)

# Velocity issues are inspected explicitly below instead of printed as warnings.
warnings.simplefilter("ignore", DesignWarning)
pd.set_option("display.max_rows", 100)

## 1. Inputs

A 51 m x 48 m greenhouse with 8 m bays, three benches per bay and five irrigation sectors.
It holds 61 grow-out benches (8 channels of the sparse `PS85` profile) and 11 nursery benches
(18 channels of the dense `PS55` profile), all 12 m x 1.8 m. The site rises 6 % along the length.

The same design is available as a TOML file in `examples/greenhouse_51x48.toml`.

In [2]:
greenhouse = Greenhouse(
    length_m=51,
    width_m=48,
    bay_width_m=8,
    benches_per_bay=3,
    sectors=5,
    min_aisle_width_m=0.53,
)
grow_out = Bench(length_m=12, width_m=1.8, channels=8, profile="PS85")
nursery = Bench(length_m=12, width_m=1.8, channels=18, profile="PS55")
benches = (
    BenchGroup(grow_out, count=61, crop="Lettuce (grow-out)"),
    BenchGroup(nursery, count=11, crop="Lettuce (nursery)"),
)
hydraulics = HydraulicSettings(main_pipe_mm=50, lateral_pipe_mm=32, terrain_slope_pct=6.0)
station = PumpStationSettings(efficiency=0.6, reservoir_volume_l=3000)

## 2. Layout

`plan_layout` checks that the benches physically fit and derives the counts the rest of the
design depends on. Infeasible inputs raise `DesignError` (for example aisles narrower than
`min_aisle_width_m`, or more benches than bench rows).

In [3]:
layout = plan_layout(greenhouse, benches)

pd.Series(
    {
        "bays": greenhouse.bays,
        "bench rows (bays x benches per bay)": greenhouse.bench_rows,
        "bench rows used": layout.bench_rows_used,
        "benches": layout.total_benches,
        "NFT channels": layout.total_channels,
        "laterals (one per 3 benches)": layout.laterals,
        "laterals per sector": layout.laterals_per_sector,
        "aisle width (m)": round(layout.aisle_width_m, 2),
    },
    name="layout",
    dtype=object,
).to_frame()

,layout
bays,6
bench rows (bays x benches per bay),18
bench rows used,18
benches,72
NFT channels,686
laterals (one per 3 benches),24
laterals per sector,5
aisle width (m),0.87


In [4]:
pd.DataFrame(
    [
        {
            "crop": g.crop,
            "benches": g.count,
            "bench": g.bench.label,
            "hole spacing (cm)": round(g.bench.profile.hole_spacing_cm, 1),
            "channel spacing (cm)": round(g.bench.channel_spacing_cm, 1),
            "plant sites per bench": g.bench.plant_sites,
            "plant sites": g.plant_sites,
        }
        for g in layout.groups
    ]
)

,crop,benches,bench,hole spacing (cm),channel spacing (cm),plant sites per bench,plant sites
0,Lettuce (grow-out),61,"12 x 1.8 m, 8 x PS85 channels",26.1,24.9,368,22448
1,Lettuce (nursery),11,"12 x 1.8 m, 18 x PS55 channels",10.3,10.3,2088,22968


What happens with a fourth bench per bay? The aisles shrink to 0.2 m and the layout is rejected:

In [5]:
from hydroponics import DesignError

try:
    plan_layout(Greenhouse(51, 48, benches_per_bay=4, sectors=5), benches)
except DesignError as exc:
    print(f"DesignError: {exc}")

DesignError: Aisle width 0.20 m is below the minimum of 0.53 m; use narrower benches, fewer benches per bay or a wider bay


## 3. Hydraulics

**Design flow.** Each NFT channel receives 1.5 L/min. The flow of a sector is the total split
between sectors, times a 1.3 safety factor.

**Friction loss** uses the Hazen-Williams equation (SI units, $Q$ in m³/s, $D$ inner diameter in m,
$C = 140$ for PVC):

$$h_f = 10.646 \, L \, \frac{(Q/C)^{1.852}}{D^{4.87}}$$

**Pipes with outlets.** A manifold or lateral loses flow at each outlet, so its real loss is lower
than for a plain pipe carrying the inlet flow all the way. The Christiansen factor $F$ corrects
this for $N$ equally spaced outlets ($m = 1.852$):

$$F = \frac{1}{m+1} + \frac{1}{2N} + \frac{\sqrt{m-1}}{6N^2}$$

**Path to the farthest bench** (worst case):

1. *Feed line*: greenhouse width + 10 m at full sector flow ($F = 1$).
2. *Manifold*: along the greenhouse length, one outlet per lateral in the sector.
3. *Lateral*: across one bay, feeding three benches of the most demanding type (nursery: 18 channels).

Fittings and valves add 10 % of the friction loss; bench height, filter, suction and the terrain
slope are added as fixed heads. Each segment's inlet velocity is checked against 0.5-2.0 m/s:
faster flow means steep friction losses and water-hammer risk, slower flow means an oversized pipe.

In [6]:
design = design_greenhouse(DesignInputs(greenhouse, benches, hydraulics, station))
hyd = design.hydraulics
print(f"Total flow (nominal):   {hyd.total_flow_m3h:.2f} m³/h")
print(f"Design flow per sector: {hyd.sector_flow_m3h:.2f} m³/h")
hyd.segments_frame()

Total flow (nominal):   61.74 m³/h
Design flow per sector: 16.05 m³/h


,segment,DN (mm),ID (mm),length (m),flow (m³/h),outlets,velocity (m/s),velocity ok,F,head loss (m)
0,Feed line,50,44.0,58.0,16.052,1,2.93,False,1.000,11.72
1,Manifold,50,44.0,51.0,16.052,5,2.93,False,0.457,4.71
2,Lateral,32,27.8,8.0,6.318,3,2.89,False,0.534,1.44


In [7]:
hyd.head_budget_frame()

,item,type,head (m)
0,Feed line,friction,11.72
1,Manifold,friction,4.71
2,Lateral,friction,1.44
3,Fittings and valves (+10% of friction),minor,1.79
4,Bench height,static / component,1.20
5,Filter,static / component,1.50
6,Suction,static / component,1.50
7,Terrain slope,static / component,3.06
8,Total dynamic head,total,26.90


In [8]:
for message in design.warnings:
    print(message)

Feed line: velocity 2.93 m/s is above 2 m/s; use DN60 or larger
Manifold: velocity 2.93 m/s is above 2 m/s; use DN60 or larger
Lateral: velocity 2.89 m/s is above 2 m/s; use DN40 or larger


All three pipes run at almost 3 m/s: the DN50 main and DN32 laterals are undersized for this
greenhouse, and friction (with its fittings allowance) makes up nearly three quarters of the pump head.

### Cross-check against the original estimate

The first version of this project lumped the feed line and manifold into a single 109 m pipe with
the Christiansen factor applied over its whole length, sized the lateral for a third of the sector
flow and applied the 10 % allowance to every head, including static ones. It reported **20.33 m**.
Re-running that lumped estimate with the package's functions reproduces it (the last digit differs
because the original rounded intermediate values):

In [9]:
sector_flow = 16.052  # m³/h, as rounded in the original
laterals_per_sector = 72 / (3 * 5)  # 4.8 on average

main = hazen_williams_head_loss(51 + 48 + 10, sector_flow, 44.0) * christiansen_factor(laterals_per_sector)
lateral = hazen_williams_head_loss(8, sector_flow / 3, 27.8) * christiansen_factor(3)
lumped = (main + lateral + 1.2 + 1.5 + 1.5 + 0.06 * 51) * 1.1
print(f"Lumped estimate: {lumped:.2f} m   |   segmented model: {hyd.total_dynamic_head_m:.2f} m")

Lumped estimate: 20.32 m   |   segmented model: 26.90 m


The segmented model is higher mainly because the 58 m feed line carries the full sector flow
before the first outlet, so the Christiansen reduction does not apply to it.

## 4. Pipe sizing trade-off

Head loss scales with $D^{-4.87}$, so one size up in the main line changes the pump considerably.
The sweep below re-runs the whole design for combinations of main and lateral diameters.

In [10]:
rows = []
for main_mm in (50, 60, 75):
    for lateral_mm in (32, 40):
        settings = HydraulicSettings(main_pipe_mm=main_mm, lateral_pipe_mm=lateral_mm, terrain_slope_pct=6.0)
        d = design_greenhouse(DesignInputs(greenhouse, benches, settings, station))
        rows.append(
            {
                "main DN": main_mm,
                "lateral DN": lateral_mm,
                "max velocity (m/s)": round(max(s.velocity_m_s for s in d.hydraulics.segments), 2),
                "velocity ok": not d.warnings,
                "TDH (m)": round(d.hydraulics.total_dynamic_head_m, 1),
                "shaft power (kW)": round(d.pump.shaft_power_w / 1000, 2),
                "motor (cv)": d.pump.motor_cv,
            }
        )
sweep = pd.DataFrame(rows)
sweep

,main DN,lateral DN,max velocity (m/s),velocity ok,TDH (m),shaft power (kW),motor (cv)
0,50,32,2.93,False,26.9,1.96,3.0
1,50,40,2.93,False,25.8,1.88,3.0
2,60,32,2.89,False,15.9,1.16,2.0
3,60,40,1.99,True,14.8,1.08,1.5
4,75,32,2.89,False,11.2,0.82,1.5
5,75,40,1.80,True,10.2,0.74,1.5


DN60 main with DN40 laterals is the smallest combination that keeps every segment inside the
velocity band (only just: 1.99 m/s in the main). It roughly halves the pump head and the motor size. DN75 cuts the head further, at the cost of larger pipe and fittings; the final choice
depends on pipe prices versus pump and energy cost, which is where a priced catalog comes in (section 6).

## 5. Pump selection

For the DN60 / DN40 option, the pump duty point per sector is the design flow at the total dynamic head.
Shaft power is $P = \rho g Q H / \eta$ with an assumed efficiency $\eta = 0.6$; the motor is the next
standard size. The final model must still be checked against a manufacturer's pump curve at this duty point.

In [11]:
final = design_greenhouse(
    DesignInputs(
        greenhouse,
        benches,
        HydraulicSettings(main_pipe_mm=60, lateral_pipe_mm=40, terrain_slope_pct=6.0),
        station,
    )
)
print(final.summary())

Greenhouse design summary
Footprint        51 m x 48 m (2,448 m²), 6 bays of 8 m, 5 sector(s)
Benches          72 in 18/18 bench rows, 686 NFT channels, 24 laterals
Aisle width      0.87 m (minimum 0.53 m)
    61 x Lettuce (grow-out)     12 x 1.8 m, 8 x PS85 channels, channel spacing 24.9 cm, 22,448 plant sites
    11 x Lettuce (nursery)      12 x 1.8 m, 18 x PS55 channels, channel spacing 10.3 cm, 22,968 plant sites
Plant sites      45,416 total

Hydraulics (per sector)
Nominal flow     61.74 m³/h for the whole greenhouse (686 channels x 1.5 L/min)
Design flow      16.05 m³/h per sector (x1.3 safety factor)
  Feed line  DN60   L= 58.0 m  Q= 16.05 m³/h  v=1.99 m/s  hf= 4.56 m  (plain pipe)
  Manifold   DN60   L= 51.0 m  Q= 16.05 m³/h  v=1.99 m/s  hf= 1.83 m  (5 outlets, F=0.457)
  Lateral    DN40   L=  8.0 m  Q=  6.32 m³/h  v=1.80 m/s  hf= 0.46 m  (3 outlets, F=0.534)
Friction         6.85 m + 0.69 m fittings
Static/component 7.26 m
Total head       14.80 m

Pump (one per sector)
Duty 

## 6. Bill of materials

Quantities come from the layout: trestles every metre, channel bars of 6 m, one clip and screw per
channel per trestle, supply pipes in 6 m bars, one pump station per sector, and consumables scaled by
floor area. Identical products from different bench types are merged into a single line.

In [12]:
final.bom_frame()

,category,product,quantity,unit
0,Benches,bench-trestle-1.8m,936,pc
1,Benches,inlet-trestle-8ch-1.8m,61,pc
2,Benches,collector-gutter-8ch-1.8m,61,pc
3,Benches,nft-channel-PS85,976,bar (6 m)
4,Benches,channel-end-cap-PS85,488,pc
5,Benches,channel-clip-PS85,6344,pc
6,Benches,screw,8918,pc
7,Benches,inlet-trestle-18ch-1.8m,11,pc
8,Benches,collector-gutter-18ch-1.8m,11,pc
9,Benches,nft-channel-PS55,396,bar (6 m)


### Pricing with a product catalog

Codes and prices are not part of the repository. To price the BOM, create a CSV with a `product`
column matching the keys above plus any of `code`, `description` and `unit_price` (`,` or `;`
delimited), then:

```python
from hydroponics import Catalog

catalog = Catalog.from_csv("path/to/catalog.csv")
priced = final.bom_frame(catalog)
priced["total_price"].sum()
```

Products missing from the catalog are kept with empty price columns and listed in a single log warning.